# A Comparison of Flare Forecasting Methods. II — Implementation
# 태양 플레어 예측 방법 비교 II — 구현

**Paper / 논문**: Leka, Park, Kusano, Andries et al. 2019, ApJS, 243, 36 — "A Comparison of Flare Forecasting Methods. II. Benchmarks, Metrics, and Performance Results for Operational Solar Flare Forecasting Systems"

**Goal / 목표 (EN)**: Reproduce in a self-contained synthetic experiment the core evaluation pipeline used by Leka+ 2019: simulate 11 forecast methods with diverse skill profiles on a 731-day testing interval mimicking 2016–2017, then compute and visualize the standard skill-score suite — TSS, HSS, CSI, BSS, MSESS_clim, ApSS, Gini, Frequency Bias — together with reliability diagrams, ROC curves, contingency tables, and persistence-vs-ML comparison.

**목표 (KO)**: Leka+ 2019에서 사용된 핵심 평가 파이프라인을 자체 포함된 합성 실험으로 재현 — 다양한 skill 프로파일을 가진 11개 예측 방법을 2016–2017을 모사한 731일 테스트 구간에서 시뮬레이션하고, 표준 skill-score 세트(TSS, HSS, CSI, BSS, MSESS_clim, ApSS, Gini, Frequency Bias)와 reliability diagram, ROC 곡선, contingency table, persistence-vs-ML 비교를 계산·시각화.

**Notebook structure / 노트북 구조**:
1. Imports & synthetic ground-truth event series / 임포트와 합성 ground-truth 이벤트 계열
2. Eleven simulated forecast methods / 11개 시뮬레이션 예측 방법
3. Skill-score functions (TSS, HSS, CSI, BSS, MSESS_clim, ApSS, Gini) / Skill-score 함수
4. Contingency tables and skill-score panel / 분할표와 skill-score 패널
5. Reliability diagrams / 신뢰도 다이어그램
6. ROC curves and AUC via np.trapezoid / ROC 곡선과 AUC
7. Persistence vs ML method comparison / Persistence vs ML 비교

## Part 1 — Imports & synthetic ground-truth event series
## Part 1 — 임포트와 합성 ground-truth 이벤트 계열

**EN**: We construct a 731-day binary event series for M1.0+/0/24 with climatology rate ≈ 0.036, matching Table 2 of the paper (26 events out of 731 days). Events are clustered (active regions cause runs of consecutive event-days), reproducing the temporal autocorrelation seen in real GOES data.

**KO**: 논문 표 2와 일치(731일 중 26 이벤트)하는 기후값 비율 ≈ 0.036의 M1.0+/0/24 731일 이진 이벤트 계열을 구성. 이벤트는 군집(활성영역이 연속 이벤트일을 일으킴)되어 실제 GOES 데이터의 시간 자기상관을 재현.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, Tuple

# Reproducibility: deterministic random seed for synthetic data.
rng = np.random.default_rng(seed=20260427)

N_DAYS = 731  # 2016-01-01 to 2017-12-31, matching Leka+ 2019 testing interval.
TARGET_M_RATE = 26.0 / 731.0  # M1.0+ event-day rate from paper Table 2.


def generate_clustered_events(n_days: int, target_rate: float, cluster_lambda: float = 3.0) -> np.ndarray:
    """Generate a binary event series with active-region clustering.

    Events arrive in clusters of geometric length to mimic flare-productive
    active regions persisting for several days.

    Args:
        n_days: Total length of the testing interval.
        target_rate: Target fraction of event-days.
        cluster_lambda: Mean cluster duration (days).

    Returns:
        Binary numpy array of shape (n_days,) with values in {0, 1}.
    """
    obs = np.zeros(n_days, dtype=int)
    target_count = int(round(target_rate * n_days))
    placed = 0
    while placed < target_count:
        start = rng.integers(0, n_days)
        length = max(1, int(rng.exponential(cluster_lambda)))
        end = min(n_days, start + length)
        for j in range(start, end):
            if obs[j] == 0 and placed < target_count:
                obs[j] = 1
                placed += 1
    return obs


obs_M = generate_clustered_events(N_DAYS, TARGET_M_RATE, cluster_lambda=3.0)
obs_C = generate_clustered_events(N_DAYS, 188.0 / 731.0, cluster_lambda=4.0)
# A C1.0+ day is also an event-day if observed; keep separate for two analyses.

print(f"Generated event series with {N_DAYS} days.")
print(f"M1.0+ event-days: {obs_M.sum()} (target ~26)")
print(f"C1.0+ event-days: {obs_C.sum()} (target ~188)")
print(f"M1.0+ climatology rate: {obs_M.mean():.4f}")
print(f"C1.0+ climatology rate: {obs_C.mean():.4f}")

## Part 2 — Eleven simulated forecast methods
## Part 2 — 11개 시뮬레이션 예측 방법

**EN**: We construct 11 simulated probabilistic forecasts spanning the skill spectrum reported in the paper. Each method has a different signal-to-noise ratio, calibration bias, and probability distribution. The list is conceptually inspired by the Table 1 cohort (DAFFS, MCSTAT, MCEVOL, MAG4 family, MOSWOC, NOAA, SIDC, NICT, NJIT, ASSA, AMOS) plus a CLIM120 baseline and a Persistence baseline.

**KO**: 논문에서 보고된 skill 스펙트럼을 망라하는 11개 시뮬레이션 확률 예보를 구성. 각 방법은 다른 신호-잡음비, 보정 bias, 확률 분포를 가짐. 표 1의 코호트(DAFFS, MCSTAT, MCEVOL, MAG4 계열, MOSWOC, NOAA, SIDC, NICT, NJIT, ASSA, AMOS)에 CLIM120 baseline과 Persistence baseline 추가.

In [ ]:
def make_signal(obs: np.ndarray, signal_strength: float, noise_sigma: float, bias: float = 0.0) -> np.ndarray:
    """Build a synthetic probabilistic forecast from a binary truth series.

    The forecast is a sigmoid of (signal_strength * obs - 0.5 * signal_strength + noise + bias),
    so high signal_strength tracks the truth tightly while noise_sigma erodes discrimination.

    Args:
        obs: Binary observation array.
        signal_strength: Latent signal amplitude (higher = better discrimination).
        noise_sigma: Standard deviation of latent noise (higher = worse).
        bias: Additive shift in latent space (positive = over-forecast).

    Returns:
        Probability forecast array p in (0, 1) of same length as obs.
    """
    latent = signal_strength * obs - 0.5 * signal_strength + rng.normal(0, noise_sigma, size=obs.shape) + bias
    p = 1.0 / (1.0 + np.exp(-latent))
    return np.clip(p, 1e-4, 1.0 - 1e-4)


def make_clim120(obs: np.ndarray) -> np.ndarray:
    """Compute the 120-day prior climatology forecast (CLIM120) for each day.

    For day t the forecast is the mean event rate over [t-120, t-1]. The first
    120 days fall back to the global climatology of the testing interval, which
    is consistent with the paper since the prior 120 days exist outside the
    testing window.
    """
    p = np.zeros_like(obs, dtype=float)
    global_rate = obs.mean()
    for t in range(len(obs)):
        lo = max(0, t - 120)
        if t == 0:
            p[t] = global_rate
        else:
            p[t] = obs[lo:t].mean() if (t - lo) > 0 else global_rate
    return np.clip(p, 1e-4, 1.0 - 1e-4)


def make_persistence(obs: np.ndarray) -> np.ndarray:
    """Persistence forecast: probability today = observation from yesterday.

    Day 0 falls back to global climatology since no prior day exists.
    """
    p = np.empty_like(obs, dtype=float)
    p[0] = obs.mean()
    p[1:] = obs[:-1].astype(float)
    # Soften 0/1 to avoid log-undefined behavior in BSS.
    return np.clip(p, 1e-4, 1.0 - 1e-4)


# Build the 11-method ensemble for M1.0+ events. Parameters tuned to span
# the skill range qualitatively reported in the paper.
methods_M: Dict[str, np.ndarray] = {
    "DAFFS":   make_signal(obs_M, signal_strength=4.5, noise_sigma=1.4, bias=-0.2),
    "MCSTAT":  make_signal(obs_M, signal_strength=4.2, noise_sigma=1.5, bias=0.0),
    "MCEVOL":  make_signal(obs_M, signal_strength=4.0, noise_sigma=1.6, bias=0.0),
    "MAG4VWF": make_signal(obs_M, signal_strength=3.8, noise_sigma=1.7, bias=0.1),
    "MOSWOC":  make_signal(obs_M, signal_strength=3.5, noise_sigma=1.8, bias=-0.3),  # human FITL
    "NOAA":    make_signal(obs_M, signal_strength=3.4, noise_sigma=1.8, bias=-0.2),  # human
    "SIDC":    make_signal(obs_M, signal_strength=3.2, noise_sigma=1.9, bias=-0.4),  # human
    "ASSA":    make_signal(obs_M, signal_strength=3.0, noise_sigma=2.0, bias=0.0),
    "AMOS":    make_signal(obs_M, signal_strength=2.8, noise_sigma=2.1, bias=0.2),
    "NJIT":    make_signal(obs_M, signal_strength=2.5, noise_sigma=2.2, bias=0.5),  # over-forecasts
    "CLIM120": make_clim120(obs_M),  # Operationally honest no-skill reference.
}

# Persistence forecast as an additional baseline for Part 7.
persistence_M = make_persistence(obs_M)

print("Method probability summaries (M1.0+):")
for name, p in methods_M.items():
    print(f"  {name:8s}  mean={p.mean():.3f}  min={p.min():.3f}  max={p.max():.3f}")

## Part 3 — Skill-score functions
## Part 3 — Skill-score 함수

**EN**: Implement the metric suite described in Section 2.2 of the paper. All metrics operate on either probability arrays or contingency tables.

**KO**: 논문 Section 2.2에서 설명된 메트릭 세트 구현. 모든 메트릭은 확률 배열 또는 분할표에서 작동.

In [ ]:
def contingency_table(p: np.ndarray, obs: np.ndarray, p_th: float = 0.5) -> Tuple[int, int, int, int]:
    """Compute the 2x2 contingency table at a given probability threshold.

    Args:
        p: Probability forecast array.
        obs: Binary observation array.
        p_th: Probability threshold for yes/no classification.

    Returns:
        Tuple (TP, FP, FN, TN).
    """
    yhat = (p >= p_th).astype(int)
    TP = int(((yhat == 1) & (obs == 1)).sum())
    FP = int(((yhat == 1) & (obs == 0)).sum())
    FN = int(((yhat == 0) & (obs == 1)).sum())
    TN = int(((yhat == 0) & (obs == 0)).sum())
    return TP, FP, FN, TN


def tss(TP: int, FP: int, FN: int, TN: int) -> float:
    """True Skill Statistic (Hanssen-Kuiper / Peirce) = POD - POFD."""
    pod = TP / max(TP + FN, 1)
    pofd = FP / max(FP + TN, 1)
    return pod - pofd


def hss(TP: int, FP: int, FN: int, TN: int) -> float:
    """Heidke Skill Score, reference = random forecast."""
    num = 2.0 * (TP * TN - FP * FN)
    den = (TP + FN) * (FN + TN) + (TP + FP) * (FP + TN)
    return num / den if den > 0 else 0.0


def csi(TP: int, FP: int, FN: int, TN: int) -> float:
    """Critical Success Index = TP / (TP + FP + FN).

    Equivalent to Threat Score; not a skill score but a popular accuracy
    measure for rare events.
    """
    den = TP + FP + FN
    return TP / den if den > 0 else 0.0


def freq_bias(TP: int, FP: int, FN: int, TN: int) -> float:
    """Frequency Bias = (TP + FP) / (TP + FN). >1 over-, <1 under-forecast."""
    den = TP + FN
    return (TP + FP) / den if den > 0 else np.nan


def proportion_correct(TP: int, FP: int, FN: int, TN: int) -> float:
    """Proportion Correct (Accuracy). Misleading for rare events."""
    return (TP + TN) / max(TP + FP + FN + TN, 1)


def brier_score(p: np.ndarray, obs: np.ndarray) -> float:
    """Brier Score = mean squared error of probability vs binary outcome."""
    return float(np.mean((p - obs) ** 2))


def bss(p: np.ndarray, obs: np.ndarray) -> float:
    """Brier Skill Score with reference = test-period climatology constant."""
    bs = brier_score(p, obs)
    s = obs.mean()
    bs_ref = s * (1.0 - s)  # BS of the constant-climatology forecast.
    return 1.0 - bs / bs_ref if bs_ref > 0 else 0.0


def msess_clim(p: np.ndarray, obs: np.ndarray, clim120: np.ndarray) -> float:
    """MSE skill score with reference = 120-day prior climatology forecast."""
    mse_method = float(np.mean((p - obs) ** 2))
    mse_ref = float(np.mean((clim120 - obs) ** 2))
    return 1.0 - mse_method / mse_ref if mse_ref > 0 else 0.0


def apss(p: np.ndarray, obs: np.ndarray, p_th: float = 0.5) -> float:
    """Appleman Skill Score (categorical). Reference = across-the-board forecast."""
    yhat = (p >= p_th).astype(int)
    n_wrong = int((yhat != obs).sum())
    s = obs.mean()
    if s > 0.5:
        # "Always yes" reference: wrong on every non-event day.
        n_wrong_ref = int((obs == 0).sum())
    else:
        # "Always no" reference: wrong on every event day.
        n_wrong_ref = int((obs == 1).sum())
    return 1.0 - n_wrong / n_wrong_ref if n_wrong_ref > 0 else 0.0


def roc_curve(p: np.ndarray, obs: np.ndarray, n_thresholds: int = 101) -> Tuple[np.ndarray, np.ndarray]:
    """Compute (POFD, POD) points across a sweep of probability thresholds.

    Args:
        p: Probability forecast array.
        obs: Binary observation array.
        n_thresholds: Number of thresholds to sweep in [0, 1].

    Returns:
        Tuple (pofd, pod) sorted in ascending pofd for AUC integration.
    """
    thresholds = np.linspace(0.0, 1.0, n_thresholds)
    pods = np.empty(n_thresholds)
    pofds = np.empty(n_thresholds)
    for k, t in enumerate(thresholds):
        TP, FP, FN, TN = contingency_table(p, obs, p_th=t)
        pods[k] = TP / max(TP + FN, 1)
        pofds[k] = FP / max(FP + TN, 1)
    # Sort by pofd ascending for trapezoidal integration.
    order = np.argsort(pofds)
    return pofds[order], pods[order]


def auc_gini(p: np.ndarray, obs: np.ndarray) -> Tuple[float, float]:
    """Compute AUC by trapezoid rule and the Gini coefficient = 2*AUC - 1."""
    pofd, pod = roc_curve(p, obs)
    auc = float(np.trapezoid(pod, pofd))
    return auc, 2.0 * auc - 1.0


# Smoke test: persistence baseline on M1.0+.
TP, FP, FN, TN = contingency_table(persistence_M, obs_M)
print(f"Persistence M1.0+ contingency: TP={TP}, FP={FP}, FN={FN}, TN={TN}")
print(f"  TSS={tss(TP,FP,FN,TN):.3f}  HSS={hss(TP,FP,FN,TN):.3f}  CSI={csi(TP,FP,FN,TN):.3f}")
auc, gini = auc_gini(persistence_M, obs_M)
print(f"  AUC={auc:.3f}  Gini={gini:.3f}")

## Part 4 — Contingency tables and skill-score panel
## Part 4 — 분할표와 skill-score 패널

**EN**: Compute every metric for every method at P_th = 0.5 (the operationally honest threshold per Section 2.2.5 of the paper). Display as a single comparison table.

**KO**: 모든 방법에 대해 P_th = 0.5(논문 Section 2.2.5의 운영적으로 정직한 임계값)에서 모든 메트릭을 계산. 단일 비교 표로 표시.

In [ ]:
clim120_M = methods_M["CLIM120"]

rows = []
for name, p in methods_M.items():
    TP, FP, FN, TN = contingency_table(p, obs_M, p_th=0.5)
    auc, gini = auc_gini(p, obs_M)
    rows.append({
        "Method": name,
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "TSS": tss(TP, FP, FN, TN),
        "HSS": hss(TP, FP, FN, TN),
        "CSI": csi(TP, FP, FN, TN),
        "FB": freq_bias(TP, FP, FN, TN),
        "PC": proportion_correct(TP, FP, FN, TN),
        "BSS": bss(p, obs_M),
        "MSESS_clim": msess_clim(p, obs_M, clim120_M),
        "ApSS": apss(p, obs_M),
        "AUC": auc,
        "Gini": gini,
    })

df = pd.DataFrame(rows)
df_disp = df.copy()
for col in ["TSS", "HSS", "CSI", "FB", "PC", "BSS", "MSESS_clim", "ApSS", "AUC", "Gini"]:
    df_disp[col] = df_disp[col].round(3)
print("Skill-score panel for M1.0+/0/24 at P_th = 0.5:")
print(df_disp.to_string(index=False))

In [ ]:
# Visualize the skill-score panel as a horizontal bar chart sorted by TSS.
metrics_to_plot = ["TSS", "HSS", "CSI", "BSS", "MSESS_clim", "ApSS", "Gini"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(20, 5), sharey=True)
df_sorted = df.sort_values("TSS", ascending=True).reset_index(drop=True)

for ax, metric in zip(axes, metrics_to_plot):
    colors = ["#d62728" if v < 0 else "#2ca02c" for v in df_sorted[metric]]
    ax.barh(df_sorted["Method"], df_sorted[metric], color=colors)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(metric)
    ax.grid(True, axis="x", alpha=0.3)

fig.suptitle("Skill-score panel — M1.0+/0/24, P_th = 0.5 (Leka+ 2019 reproduction)", fontsize=13)
fig.tight_layout()
plt.show()

**Interpretation / 해석**

**EN**: Notice how the ranking shifts depending on which metric is examined. Methods strong on TSS may be weaker on BSS (because TSS rewards discrimination at one threshold while BSS rewards calibration across the entire probability range). CLIM120 by construction sits at MSESS_clim ≈ 0 — confirming the metric works as a sanity check. Some methods score below 0 on selected metrics, illustrating the paper's central finding that being "operational" does not guarantee skill on every metric.

**KO**: 어떤 메트릭을 보느냐에 따라 순위가 어떻게 변하는지 주목. TSS에 강한 방법이 BSS에 약할 수 있음(TSS는 단일 임계값에서 판별을 보상하지만 BSS는 확률 전 범위에서 보정을 보상). CLIM120은 구조상 MSESS_clim ≈ 0 — sanity check로 메트릭이 작동함을 확인. 일부 방법은 일부 메트릭에서 0 미만 점수 — "운영 중"이라는 사실이 모든 메트릭에서 skill을 보장하지 않는다는 논문의 핵심 발견 예시.

## Part 5 — Reliability diagrams
## Part 5 — 신뢰도 다이어그램

**EN**: For each method, group days into 10 probability bins and plot mean predicted probability vs observed event frequency. The 1:1 line is perfect calibration; the climatology horizontal line and the no-skill bisector are also drawn (Figure 2 of paper).

**KO**: 각 방법에 대해 일자를 10개 확률 bin으로 그룹화하고 평균 예측 확률 vs 관측 이벤트 빈도 plot. 1:1 선이 완벽 보정; 기후값 수평선과 no-skill 이등분선도 그림(논문 그림 2).

In [ ]:
def reliability_bins(p: np.ndarray, obs: np.ndarray, n_bins: int = 10) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bin probabilities and compute mean predicted vs observed frequency per bin.

    Returns:
        (bin_centers, observed_freq, count_per_bin) where empty bins are NaN.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    obs_freq = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)
    for b in range(n_bins):
        mask = (p >= edges[b]) & (p < edges[b + 1] if b < n_bins - 1 else p <= edges[b + 1])
        counts[b] = int(mask.sum())
        if counts[b] > 0:
            obs_freq[b] = obs[mask].mean()
    return centers, obs_freq, counts


method_names = list(methods_M.keys())
n_methods = len(method_names)
n_cols = 4
n_rows = int(np.ceil(n_methods / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.4 * n_rows), sharex=True, sharey=True)
axes = axes.flatten()

clim_rate = obs_M.mean()
no_skill_line_x = np.linspace(0, 1, 50)
no_skill_line_y = 0.5 * (no_skill_line_x + clim_rate)  # bisector between climatology and y=x.

for idx, name in enumerate(method_names):
    ax = axes[idx]
    centers, obs_freq, counts = reliability_bins(methods_M[name], obs_M, n_bins=10)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect")
    ax.axhline(clim_rate, color="gray", lw=0.8, ls=":", label="Clim")
    ax.plot(no_skill_line_x, no_skill_line_y, color="gray", lw=0.8, ls="-.", label="No-skill")
    ax.plot(centers, obs_freq, "o-", color="C0", lw=1.5, markersize=5, label=name)
    # Show population fraction as small red squares (Leka+ 2019 style).
    pop_frac = counts / counts.sum() if counts.sum() > 0 else counts
    ax.plot(centers, pop_frac, "s", color="red", markersize=3, alpha=0.7)
    ax.set_title(name)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

for idx in range(n_methods, len(axes)):
    axes[idx].set_visible(False)

fig.text(0.5, -0.01, "Predicted probability", ha="center", fontsize=12)
fig.text(-0.01, 0.5, "Observed frequency / Population fraction (red)", va="center", rotation="vertical", fontsize=12)
fig.suptitle("Reliability diagrams — M1.0+/0/24 (cf. Leka+ 2019 Figure 2)", fontsize=13)
fig.tight_layout()
plt.show()

## Part 6 — ROC curves and AUC via np.trapezoid
## Part 6 — ROC 곡선과 AUC (np.trapezoid 사용)

**EN**: Plot ROC curves (POD vs POFD) for all 11 methods on a single panel. The diagonal x=y is the no-skill line. AUC is computed by trapezoidal integration. Compare to Figure 3 of the paper.

**KO**: 모든 11개 방법의 ROC 곡선(POD vs POFD)을 단일 패널에 plot. 대각선 x=y가 no-skill 선. AUC는 사다리꼴 적분으로 계산. 논문 그림 3과 비교.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="No-skill (x=y)")

cmap = plt.colormaps["tab20"]
auc_records = []
for i, (name, p) in enumerate(methods_M.items()):
    pofd, pod = roc_curve(p, obs_M)
    auc = float(np.trapezoid(pod, pofd))
    auc_records.append((name, auc))
    ax.plot(pofd, pod, "-", color=cmap(i % 20), lw=1.6, label=f"{name} (AUC={auc:.2f})")

ax.set_xlabel("Probability of False Detection (POFD)")
ax.set_ylabel("Probability of Detection (POD)")
ax.set_title("ROC curves — M1.0+/0/24 (cf. Leka+ 2019 Figure 3)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAUC ranking (higher = better discrimination):")
for name, auc in sorted(auc_records, key=lambda x: -x[1]):
    print(f"  {name:8s}  AUC = {auc:.3f}  Gini = {2*auc - 1:+.3f}")

## Part 7 — Persistence vs ML method comparison
## Part 7 — Persistence vs ML 방법 비교

**EN**: A trivial "persistence" baseline (today = yesterday) is a tougher reference than CLIM120 because it captures short-term active-region clustering. We compare three references — CLIM120, Persistence, and a constant-climatology forecast — against our best ML-style method (DAFFS). This illustrates the paper's argument that the choice of reference forecast strongly affects skill scores.

**KO**: 자명한 "persistence" 기준선(오늘 = 어제)은 단기 활성영역 군집을 포착하므로 CLIM120보다 더 까다로운 기준. 세 기준(CLIM120, Persistence, 상수 기후값)을 최고 ML 스타일 방법(DAFFS)과 비교. 기준 예보 선택이 skill score에 강한 영향을 미친다는 논문의 주장을 설명.

In [ ]:
constant_clim_M = np.full_like(obs_M, fill_value=obs_M.mean(), dtype=float)

comparison = {
    "Constant climatology": constant_clim_M,
    "CLIM120": clim120_M,
    "Persistence": persistence_M,
    "DAFFS (ML-style)": methods_M["DAFFS"],
}

rows2 = []
for name, p in comparison.items():
    TP, FP, FN, TN = contingency_table(p, obs_M, p_th=0.5)
    auc, gini = auc_gini(p, obs_M)
    rows2.append({
        "Method": name,
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "TSS": round(tss(TP, FP, FN, TN), 3),
        "HSS": round(hss(TP, FP, FN, TN), 3),
        "CSI": round(csi(TP, FP, FN, TN), 3),
        "BSS": round(bss(p, obs_M), 3),
        "MSESS_clim": round(msess_clim(p, obs_M, clim120_M), 3),
        "AUC": round(auc, 3),
    })

df_cmp = pd.DataFrame(rows2)
print("Persistence vs ML comparison (M1.0+/0/24):")
print(df_cmp.to_string(index=False))

In [ ]:
# Bar plot comparing the four reference/method choices on multiple metrics.
metrics = ["TSS", "HSS", "CSI", "BSS", "MSESS_clim", "AUC"]
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(metrics))
width = 0.18
for i, name in enumerate(df_cmp["Method"]):
    vals = df_cmp.loc[df_cmp["Method"] == name, metrics].values.flatten()
    ax.bar(x + (i - 1.5) * width, vals, width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Score")
ax.set_title("Persistence vs CLIM120 vs Constant-climatology vs ML-style — M1.0+/0/24")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

**EN**: This notebook reproduces the core evaluation pipeline of Leka+ 2019 on synthetic data:
1. A 731-day clustered binary event series with M1.0+ rate ≈ 0.036 (matching paper Table 2).
2. Eleven simulated forecast methods spanning the skill spectrum, plus persistence and constant-climatology baselines.
3. A complete skill-score implementation (TSS, HSS, CSI, BSS, MSESS_clim, ApSS, Frequency Bias, AUC/Gini) with contingency-table decomposition.
4. Reliability diagrams (calibration), ROC curves (discrimination), and a unified comparison table.
5. Demonstration that **method ranking depends on the metric chosen**, reproducing the paper's central finding.
6. Demonstration that **persistence is a tougher baseline than CLIM120** — operational forecasts must beat short-timescale autocorrelation, not just long-term climatology.

**KO**: 이 노트북은 합성 데이터에서 Leka+ 2019의 핵심 평가 파이프라인을 재현:
1. M1.0+ 비율 ≈ 0.036(논문 표 2와 일치)인 731일 군집 이진 이벤트 계열.
2. skill 스펙트럼을 망라하는 11개 시뮬레이션 예보 방법 + persistence/상수 기후값 baseline.
3. 분할표 분해를 포함한 완전한 skill-score 구현(TSS, HSS, CSI, BSS, MSESS_clim, ApSS, Frequency Bias, AUC/Gini).
4. Reliability diagram(보정), ROC 곡선(판별), 통합 비교 표.
5. **선택한 메트릭에 따라 방법 순위가 달라짐**을 시연 — 논문의 핵심 발견 재현.
6. **Persistence가 CLIM120보다 까다로운 baseline임**을 시연 — 운영 예보는 장기 기후값뿐 아니라 단기 자기상관을 극복해야 함.